In [14]:
import os

os.environ["HTTP_PROXY"] = ''
os.environ["HTTPS_PROXY"] = ''
os.environ["all_proxy"] = ''
os.environ["ALL_PROXY"] = ''

In [15]:
from langchain_ollama import ChatOllama

model = ChatOllama(
    model="llama3.1",
    # temperature=0,
    # other params...
)

In [16]:
from langchain_core.messages import HumanMessage

model.invoke([HumanMessage(content="Hi! I'm Bob")])

RemoteProtocolError: Server disconnected without sending a response.

In [ ]:
from langchain_core.messages import AIMessage

model.invoke(
    [
        HumanMessage(content="Hi! I'm Bob"),
        AIMessage(content="Hello Bob! How can I assist you today?"),
        HumanMessage(content="What's my name?"),
    ]
)

# Message History

In [33]:
from langchain_core.chat_history import (
    BaseChatMessageHistory,
    InMemoryChatMessageHistory,
)
from langchain_core.runnables.history import RunnableWithMessageHistory

store = {}


def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]


with_message_history = RunnableWithMessageHistory(model, get_session_history)

In [34]:
config = {"configurable": {"session_id": "abc2"}}

In [35]:
response = with_message_history.invoke(
    [HumanMessage(content="Hi! I'm Bob")],
    config=config,
)

response.content

"Hi Bob! It's nice to meet you. Is there something on your mind that you'd like to chat about, or are you just looking for some friendly conversation? I'm here to listen and help if I can!"

In [36]:
response = with_message_history.invoke(
    [HumanMessage(content="What's my name?")],
    config=config,
)

response.content

"You told me earlier, Bob! You're Bob!"

# 提示模板

In [62]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assistant. Answer all questions to the best of your ability.",
        ),
        MessagesPlaceholder(variable_name="messages"),
    ]
)

chain = prompt | model

In [63]:
response = chain.invoke({"messages": [HumanMessage(content="hi! I'm bob")]})

response.content

"Hi Bob! It's great to meet you! I'm here to help answer any questions or provide assistance you may need. What's on your mind today? Do you have a specific topic in mind or just looking for some general chat?"

In [64]:
with_message_history = RunnableWithMessageHistory(chain, get_session_history)

In [65]:
config = {"configurable": {"session_id": "abc5"}}

In [66]:
response = with_message_history.invoke(
    [HumanMessage(content="Hi! I'm Jim")],
    config=config,
)

response.content

"We're stuck in a loop, it seems! Anyway, yes, your name is indeed Jim."

In [67]:
response = with_message_history.invoke(
    [HumanMessage(content="What's my name?")],
    config=config,
)

response.content

'Jim! (Still the same, haha!)'

In [68]:
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assistant. Answer all questions to the best of your ability in {language}.",
        ),
        MessagesPlaceholder(variable_name="messages"),
    ]
)

chain = prompt | model

In [70]:
response = chain.invoke(
    {"messages": [HumanMessage(content="hi! I'm bob")], "language": "Chinese"}
)

response.content

'😊\n\nnǐ hǎo, bōb! (你好，鲍布！) Hello, Bob! 😊'

In [75]:
with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages",
)

In [76]:
config = {"configurable": {"session_id": "abc11"}}

In [77]:
response = with_message_history.invoke(
    {"messages": [HumanMessage(content="hi! I'm todd")], "language": "Chinese"},
    config=config,
)

response.content

'😊 Ni hao, Todd! (你好，Todd!) Wang jing shen me ne? (望精神め?) How are you today? 🤗'

In [ ]:
response = with_message_history.invoke(
    {"messages": [HumanMessage(content="whats my name?")], "language": "Spanish"},
    config=config,
)

response.content